## 去除不同cancer中的肿瘤细胞，对剩下的normal cells进行合并，包括gene expr和peak matrix重构
## 建议直接在R中运行，极其占用内存

In [1]:
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)
library(Signac)
library(tibble)
library(future)
library(GenomicRanges)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86) 

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘patchwork’ was built under R version 4.3.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.fra

In [2]:
setwd("/mnt/netshare2/miaoyuanyuan/work/4_cancer_try/1_cancer11/1_merge/mergeall/")

files <- list.files(path = "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data", 
                    pattern = "annoAll\\.rds$",  # 使用正则表达式匹配文件名结尾
                    recursive = TRUE,            # 相当于 /*/，让它去所有子文件夹里翻
                    full.names = TRUE)           # 必须加这个！否则只返回文件名，没有绝对路径，后面 readRDS 会报错
files

[1] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/BRCA/BRCA_annoAll.rds"  
[2] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/CEAD/CEAD_annoAll.rds"  
[3] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/CESC/CESC_annoAll.rds"  
[4] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/CRC/CRC_annoAll.rds"    
[5] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/HNSCC/HNSCC_annoAll.rds"
[6] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/OV/OV_annoAll.rds"      
[7] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/PDAC/PDAC_annoAll.rds"  
[8] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/SKCM/SKCM_annoAll.rds"  
[9] "/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data/UCEC/UCEC_annoAll.rds"

In [3]:
cancer_names <- sapply(strsplit(basename(files), "_"), `[`, 1)
cancer_names

[1] "BRCA"  "CEAD"  "CESC"  "CRC"   "HNSCC" "OV"    "PDAC"  "SKCM"  "UCEC"

In [4]:
seurat_list <- lapply(files, readRDS)
names(seurat_list) <- cancer_names

In [5]:
seurat_list

$BRCA
An object of class Seurat 
866810 features across 72587 samples within 4 assays 
Active assay: RNA (36601 features, 0 variable features)
 14 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8, counts.9, counts.10, counts.11, counts.12, counts.13, counts.14
 3 other assays present: ATAC, peaks, SCT
 3 dimensional reductions calculated: pca, lsi, wnn.umap.unint

$CEAD
An object of class Seurat 
575830 features across 8156 samples within 4 assays 
Active assay: SCT (24690 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CESC
An object of class Seurat 
749687 features across 30100 samples within 4 assays 
Active assay: SCT (26699 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CRC
An 

In [12]:
seurat_list[[1]]@meta.data

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ATAC,nFeature_ATAC,doublet_RNA,doublet_ATAC,predicted_doublet_final,percent.mt,nucleosome_signal,⋯,nCount_SCT,nFeature_SCT,SCT.weight,peaks.weight,SCT_snn_res.0.5,seurat_clusters,sample,cancer_type,wsnn.unint_res.0.1,celltype
,<chr>,<dbl>,<int>,<dbl>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<dbl>,<chr>,<fct>,<chr>,<chr>,<fct>,<chr>
HT235B1-S1H1_AAACAGCCAAGGTCCT-1,HT235B1-S1H1,4145,1718,13221,6134,False,False,False,0.2653800,0.6805625,⋯,3486,1701,0.604414850,0.3955852,2,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACAGCCAAGTGTTT-1,HT235B1-S1H1,3927,1776,14520,6615,False,False,False,0.2546473,0.5042963,⋯,3444,1751,0.335872777,0.6641272,1,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACAGCCATAGCGAG-1,HT235B1-S1H1,2467,1263,1088,569,False,False,False,0.4053506,0.4131274,⋯,2721,1252,0.413953674,0.5860463,5,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACAGCCATGAGTTT-1,HT235B1-S1H1,4443,1576,9300,4439,False,False,False,0.1575512,0.6035042,⋯,3463,1557,0.333102240,0.6668978,1,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACAGCCATGATTGT-1,HT235B1-S1H1,3567,1464,6063,2913,False,False,False,0.9812167,0.5390226,⋯,3318,1444,0.454528206,0.5454718,0,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACATGCAACACTTG-1,HT235B1-S1H1,3174,1449,12263,5829,False,False,False,0.3465658,0.6972892,⋯,3136,1434,0.599899016,0.4001010,0,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACATGCAACTGGGA-1,HT235B1-S1H1,3203,1572,4767,2367,False,False,False,0.3434280,0.5130435,⋯,3148,1555,0.469325666,0.5306743,2,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACATGCAGTTGCGT-1,HT235B1-S1H1,1243,759,3126,1586,False,False,False,2.5744167,0.4800550,⋯,2488,786,0.349049637,0.6509504,3,3,HT235B1_S1H1,BRCA,3,Tumor
HT235B1-S1H1_AAACATGCATGGCCCA-1,HT235B1-S1H1,3274,1440,9814,4555,False,False,False,7.2693952,0.5139470,⋯,3195,1425,0.518536209,0.4814638,4,3,HT235B1_S1H1,BRCA,3,Tumor


#### BRCA 和 HNSCC 的所有样本都是Tumor，没有Met或Normal

In [26]:
seurat_list[[1]]$type <- "Tumor"
seurat_list[[5]]$type <- "Tumor"

In [27]:
table(seurat_list[[1]]$type)


Tumor 
72587 

In [28]:
table(seurat_list[[5]]$type)


Tumor 
64096 

In [29]:
# 统计每个对象里 type 为 "Tumor" 的细胞数量（而不是Met）
tumor_counts <- sapply(names(seurat_list), function(cancer) {
  obj <- seurat_list[[cancer]]
  # 统计 type 列中等于 "Tumor" 的数量，加上 na.rm 防止有 NA 报错
  tumor_num <- sum(obj$type == "Tumor", na.rm = TRUE)
  total_num <- ncol(obj) # 当前癌症的总细胞数
  
  # 顺便算个比例
  ratio <- round((tumor_num / total_num) * 100, 2)
  
  return(paste0(tumor_num, " 个 (占比 ", ratio, "%)"))
})

# 打印统计结果
print(data.frame(Tumor_Count = tumor_counts))

                 Tumor_Count
BRCA    72587 个 (占比 100%)
CEAD     8156 个 (占比 100%)
CESC    30100 个 (占比 100%)
CRC     2770 个 (占比 3.35%)
HNSCC   64096 个 (占比 100%)
OV    30736 个 (占比 91.05%)
PDAC  70156 个 (占比 61.71%)
SKCM  15224 个 (占比 21.46%)
UCEC   8864 个 (占比 64.28%)


In [30]:
tumor_cell_counts <- sapply(names(seurat_list), function(cancer) {
  obj <- seurat_list[[cancer]]
  
  # 确保 celltype 这列存在（防御性检查）
  if ("celltype" %in% colnames(obj@meta.data)) {
    # 统计 celltype 为 "Tumor" 的细胞数量
    tumor_num <- sum(obj$celltype == "Tumor", na.rm = TRUE)
  } else {
    tumor_num <- 0
  }
  
  total_num <- ncol(obj) # 当前癌症的总细胞数
  ratio <- round((tumor_num / total_num) * 100, 2)
  
  return(paste0(tumor_num, " 个 (占比 ", ratio, "%)"))
})

# 打印真正的肿瘤细胞统计结果
print(data.frame(Tumor_Cell_Count = tumor_cell_counts))

            Tumor_Cell_Count
BRCA  46664 个 (占比 64.29%)
CEAD   7564 个 (占比 92.74%)
CESC  23751 个 (占比 78.91%)
CRC   48544 个 (占比 58.77%)
HNSCC 34182 个 (占比 53.33%)
OV    24863 个 (占比 73.65%)
PDAC  41412 个 (占比 36.43%)
SKCM  33140 个 (占比 46.71%)
UCEC  12430 个 (占比 90.14%)


In [31]:
cat("开始剔除恶性肿瘤细胞...\n")

# 遍历列表，只保留 celltype 不是 "Tumor" 的细胞
seurat_list_n <- lapply(seurat_list, function(obj) {
  subset(obj, subset = celltype != "Tumor")
})

cat("微环境细胞清洗完毕！\n")

开始剔除恶性肿瘤细胞...
微环境细胞清洗完毕！


In [32]:
seurat_list_n

$BRCA
An object of class Seurat 
866810 features across 25923 samples within 4 assays 
Active assay: RNA (36601 features, 0 variable features)
 14 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8, counts.9, counts.10, counts.11, counts.12, counts.13, counts.14
 3 other assays present: ATAC, peaks, SCT
 3 dimensional reductions calculated: pca, lsi, wnn.umap.unint

$CEAD
An object of class Seurat 
575830 features across 592 samples within 4 assays 
Active assay: SCT (24690 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CESC
An object of class Seurat 
749687 features across 6349 samples within 4 assays 
Active assay: SCT (26699 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CRC
An ob

In [35]:
celltype_counts <- lapply(seurat_list_n, function(obj) {
  table(obj$celltype)
})

In [36]:
celltype_counts

$BRCA

                  B-cells   Breast basal progenitor     Breast luminal mature 
                      820                      1067                       956 
Breast luminal progenitor                        DC               Endothelial 
                     1549                       357                       609 
              Fibroblasts               Macrophages                 Pericytes 
                     3768                      7582                       466 
             Plasma cells                   T cells                   Unknown 
                     2307                      6150                       292 

$CEAD

            Endothelial             Fibroblasts             Macrophages 
                    123                     158                     170 
Normal Epithelial cells            Plasma cells 
                     63                      78 

$CESC

                     DC             Endothelial             Fibroblasts 
                     29     

In [ ]:
# 1. 提取合并所有 Peaks
data_list <- lapply(seurat_list_n, function(x) {
  DefaultAssay(x) <- "peaks"
  return(x)
})
combined.peaks <- UnifyPeaks(object.list = data_list, mode = "reduce")
peakwidths <- width(combined.peaks)
combined.peaks <- combined.peaks[peakwidths < 10000 & peakwidths > 20]
saveRDS(combined.peaks, "./combined.peaks.rds")

In [ ]:
# 2. 多线程计算 Counts矩阵
options(future.globals.maxSize = 100 * 1024^3)
plan("multisession", workers = 16)

cat("多线程已就位，开始极速回溯计算 Counts...\n")
count.list <- lapply(data_list, function(obj) {
  FeatureMatrix(
    fragments = Fragments(obj),
    features = combined.peaks,
    cells = colnames(obj)
  )
})   ## 很耗时
# Extracting reads overlapping genomic regions
plan("sequential")
saveRDS(count.list, file = "./results/PanCancer_ATAC_count_list.rds")

In [ ]:
# 3. 提取注释信息
annotation <- GetGRangesFromEnsDb(ensdb = EnsDb.Hsapiens.v86)
seqlevels(annotation) <- paste0("chr", seqlevels(annotation))
genome(annotation) <- "hg38"

In [ ]:
library(Signac)

meta_list <- list()
master_frag_list <- list() # 用于存放泛癌所有的 Fragment 对象

for (i in seq_along(cancer_names)) {
  cancer <- cancer_names[i]
  
  # 获取当前 Seurat 对象中实际存在的旧细胞名
  old_cells_in_seurat <- colnames(data_list[[i]])
  new_cells_in_seurat <- paste0(cancer, "_", old_cells_in_seurat)
  
  # 1. 正常修改 count 矩阵列名和 meta 行名 (如果 count.list 已经算好了的话)
  if (exists("count.list") && length(count.list) >= i) {
    colnames(count.list[[i]]) <- new_cells_in_seurat
  }
  
  meta <- data_list[[i]]@meta.data
  rownames(meta) <- new_cells_in_seurat
  meta_list[[i]] <- meta
  
  # 2. 核心修复：遍历该癌症下的 *所有* Fragment 对象 (比如 BRCA 的 14 个)
  cancer_frags <- Fragments(data_list[[i]])
  
  cat(sprintf("[%s] 正在处理 %d 个 Fragment 文件...\n", cancer, length(cancer_frags)))
  
  for (j in seq_along(cancer_frags)) {
    orig_frag <- cancer_frags[[j]]
    old_cells_in_frag <- Cells(orig_frag)
    
    # a. 只保留当前 Seurat 对象里经过质控存活下来的细胞
    valid_old_cells <- old_cells_in_frag[old_cells_in_frag %in% old_cells_in_seurat]
    
    if (length(valid_old_cells) > 0) {
      # b. 生成对应的新名字 (加上癌症前缀)
      new_cells_for_frag <- paste0(cancer, "_", valid_old_cells)
      
      # c. 构造字典: 名字(names)是新的 Seurat 细胞名，值(values)是底层文件认识的旧细胞名
      mapping_dict <- valid_old_cells
      names(mapping_dict) <- new_cells_for_frag
      
      # d. 创建新的 Fragment 对象，并关闭验证绕过死板的报错机制
      new_frag <- CreateFragmentObject(
        path = orig_frag@path,
        cells = mapping_dict,
        validate.fragments = FALSE 
      )
      
      # e. 追加到全库列表中
      master_frag_list <- append(master_frag_list, new_frag)
    }
  }
}

cat(sprintf("完美！已成功提取并修复全部 %d 个 Fragment 对象！\n", length(master_frag_list)))

In [ ]:
# 5. 合并 Meta 和 ATAC 矩阵
big_meta <- lapply(meta_list, function(df) {
  as.data.frame(df) %>% rownames_to_column(var = "Cell_ID")
}) %>% 
  bind_rows() %>% 
  column_to_rownames(var = "Cell_ID")

big_counts <- do.call(cbind, count.list)

# 创建带有 Fragments 信息的全能 ChromatinAssay
chrom_assay <- CreateChromatinAssay(
  counts = big_counts,
  sep = c("-", "-"),  
  # 不再输入字符 'hg38' 触发联网下载，而是直接提取本地已有的染色体信息！
  genome = seqinfo(annotation), 
  annotation = annotation,
  fragments = master_frag_list  
)

In [ ]:
# 6. 合并 RNA 矩阵
rna_counts_list <- lapply(seq_along(seurat_list_n), function(i) {
  obj <- seurat_list_n[[i]]
  DefaultAssay(obj) <- "RNA"
  
  if (length(Layers(obj, search = "counts")) > 1) {
    obj <- JoinLayers(obj)
  }
  
  mat <- GetAssayData(obj, layer = "counts") 
  colnames(mat) <- paste0(cancer_names[i], "_", colnames(mat))
  return(mat)
})

big_rna <- do.call(cbind, rna_counts_list)

In [ ]:
# 7. 最终组装
PanCancer_TME <- CreateSeuratObject(
  counts = big_rna,
  assay = "RNA",
  meta.data = big_meta,
  project = "PanCancer_Multiome"
)


# > PanCancer_TME
# An object of class Seurat
# 36601 features across 217173 samples within 1 assay
# Active assay: RNA (36601 features, 0 variable features)
#  1 layer present: counts


PanCancer_TME[["ATAC"]] <- chrom_assay  
PanCancer_TME$pancancer_type <- sub("_.*", "", colnames(PanCancer_TME))

# 按照 V5 的标准切分 RNA Layer 以备 SCTransform
PanCancer_TME[["RNA"]] <- split(PanCancer_TME[["RNA"]], f = PanCancer_TME$sample)

saveRDS(PanCancer_TME, "./results/pancancer_merge_icldFrgm.rds")

In [ ]:
> table(PanCancer_TME$type)

   Met  Tumor
 88497 128676

> table(PanCancer_TME$pancancer_type) 

 BRCA  CEAD  CESC   CRC HNSCC    OV  PDAC  SKCM  UCEC
25923   592  6349 34054 29914  8894 72274 37814  1359

> table(PanCancer_TME$celltype)  ## 这里是每个癌症单独的注释结果

                   Acinar                 Adipocyte                  Alveolar
                     4898                        50                      1942
                  B-cells   Breast basal progenitor     Breast luminal mature
                     6989                      1067                       956
Breast luminal progenitor            Cholangiocytes                        DC
                     1549                       470                      2507
              Endothelial          Epithelial cells               Fibroblasts
                     8569                      2176                     32214
              Hepatocytes                     Islet             Keratinocytes
                     4331                      5250                       772
              Low quality               Macrophages                      Mast
                      521                     56871                       616
 Normal cells endometrium       Normal Ductal-like1   Normal Epithelial cells
                      112                     11572                       520
    Normal Squamous cells                     Other                 Pericytes
                      863                      1046                      1244
             Plasma cells           Skeletal_muscle                   T cells
                    13411                       193                     49473
                  Unknown                   UnKnown                    Unkown
                     4977                       548                        60
                    vSMCs
                     1406

> PanCancer_TME$celltype[PanCancer_TME$celltype %in% c("UnKnown", "Unkown")] <- "Unknown"
> table(PanCancer_TME$celltype)

                   Acinar                 Adipocyte                  Alveolar
                     4898                        50                      1942
                  B-cells   Breast basal progenitor     Breast luminal mature
                     6989                      1067                       956
Breast luminal progenitor            Cholangiocytes                        DC
                     1549                       470                      2507
              Endothelial          Epithelial cells               Fibroblasts
                     8569                      2176                     32214
              Hepatocytes                     Islet             Keratinocytes
                     4331                      5250                       772
              Low quality               Macrophages                      Mast
                      521                     56871                       616
 Normal cells endometrium       Normal Ductal-like1   Normal Epithelial cells
                      112                     11572                       520
    Normal Squamous cells                     Other                 Pericytes
                      863                      1046                      1244
             Plasma cells           Skeletal_muscle                   T cells
                    13411                       193                     49473
                  Unknown                     vSMCs
                     5585                      1406

### 后续去批次和降维可自行执行-仅供参考

In [ ]:
cancer <- PanCancer_TME

# ==========================================
# 阶段一：RNA 转录组降维 (SCTransform + PCA)
# ==========================================
cat("正在执行 RNA 侧的 SCTransform 归一化...\n")
DefaultAssay(cancer) <- "RNA"

library(future)
options(future.globals.maxSize = 100 * 1024^3)
# 召唤多线程
plan("multisession", workers = 8)
# 4. 运行 SCTransform
cancer <- SCTransform(cancer, 
                      vars.to.regress = c("nCount_RNA", "percent.mt"), 
                      return.only.var.genes = TRUE) 
## 这里是根据每个样本单独进行的,需要标准化每个样本的测序深度
plan("sequential")

cat("正在执行 RNA 侧的 PCA 降维...\n")
cancer <- RunPCA(cancer, assay = "SCT")
## 这里是统一一起计算的,会把刚才独立算好的各个样本的 scale.data 里的那 3000 个高变基因，在内存里瞬间“缝合”成一个全局的大矩阵。
# 它在这个缝合好的大矩阵上跑 PCA 算法，得出一个统一的、包含所有细胞的 PCA 坐标空间（也就是你最终看到的 cancer@reductions$pca）。
# 算完 PCA 坐标后，为了省内存，它立刻把那个临时缝合的大矩阵给销毁了！原对象依然保持着分层状态。

cancer <- RenameAssays(object = cancer, ATAC = 'peaks')
# ==========================================
# 阶段二：Peaks 染色质降维 (TF-IDF + SVD)
# ==========================================
cat("正在执行 Peaks 侧的标准化与 SVD 降维...\n")

DefaultAssay(cancer) <- "peaks"  # 完美贴合你的习惯！

cancer <- FindTopFeatures(cancer, min.cutoff = 5)
cancer <- RunTFIDF(cancer)
cancer <- RunSVD(cancer)

cat("双线降维全部完成！\n")

In [ ]:
saveRDS(cancer,"./results/pancancer_preprocess_PCA_SVD.rds")
# cancer <- readRDS('pancancer_preprocess_PCA_SVD.rds')

In [ ]:
library(harmony)
library(ggplot2)

DefaultAssay(cancer) <- "RNA"
cancer<-RunHarmony(cancer,
                   group.by.vars="sample",
                   reduction.use = 'pca',
                   reduction.save = "harmony1",
) 

# 使用 peaks assay，基于 LSI 空间对 ATAC 数据做批次校正
DefaultAssay(cancer) <- "peaks"
cancer<-RunHarmony(cancer,
                   group.by.vars="sample",
                   reduction.use = 'lsi',
                   reduction.save = "harmony2",
                   project.dim = FALSE
) 

# 使用 Harmony 处理后的 RNA (harmony1) 和 ATAC (harmony2) 降维结果构建多模态邻接图
cancer <- FindMultiModalNeighbors(
  object = cancer,
  reduction.list = list("harmony1", "harmony2"), 
  dims.list = list(1:30, 2:30),
  modality.weight.name = "RNA.weight",
  verbose = TRUE
)
cancer <- RunUMAP(
  object = cancer,
  nn.name = "weighted.nn",
    reduction.name = "wnn.harmony.umap", # 起了个新名字
  reduction.key = "wnnHUMAP_",
  verbose = TRUE
)
cancer<- FindClusters(
  cancer, 
  graph.name = "wsnn", 
  algorithm = 3,
  resolution = 0.5,
  verbose = FALSE
)

In [ ]:
cat("保存最终的多模态去批次对象...\n")
saveRDS(cancer, file = "PanCancer_WNN_Harmony-sample_Done.rds", compress = FALSE)

In [ ]:
> table(cancer$wsnn_res.0.5)[order(as.numeric(names(table(cancer$wsnn_res.0.5))))]

    0     1     2     3     4     5     6     7     8     9    10    11    12
25013 23895 22748 21936 16810 16757 11927  7929  7871  7780  6590  6585  6280
   13    14    15    16    17    18    19    20    21    22    23    24    25
 5007  4560  4190  3880  3717  2212  2085  1721  1258  1138   828   814   584
   26    27    28    29    30    31    32    33    34    35    36    37    38
 2828     4     4     3     3     3     3     3     3     3     3     2     2
   39    40    41    42    43    44    45    46    47    48    49    50    51
    2     2     2     2     2     2     2     2     2     2     2     2     2
   52    53    54    55    56    57    58    59    60    61    62    63    64
    2     2     2     2     2     2     2     2     2     2     2     2     2
   65    66    67    68    69    70    71    72    73    74    75    76    77
    2     2     2     2     2     2     2     2     2     2     2     2     2
   78    79    80    81    82    83    84    85    86    87    88    89    90
    2     2     2     2     2     2     2     2     2     2     2     2     2
   91    92    93    94    95    96    97    98    99   100   101   102   103
    2     2     2     2     2     2     2     2     2     2     2     2     2
  104   105   106   107   108   109   110   111   112   113   114   115   116
    2     2     2     2     2     2     2     2     2     2     2     2     2
  117   118   119   120   121   122   123   124   125   126   127   128   129
    2     2     2     2     2     2     2     2     2     2     2     2     2
  130   131   132   133   134   135
    2     2     2     2     2     2


In [ ]:
table(cancer$wsnn_res.0.8)[order(as.numeric(names(table(cancer$wsnn_res.0.8))))]

In [ ]:
# 1. 看聚类结果
p1 <- DimPlot(cancer, reduction = "wnn.harmony.umap", group.by = "seurat_clusters", label = TRUE, repel = TRUE) + 
      ggtitle("WNN Clusters (Resolution = 0.5)")

# 2. 检验去批次效果：看 9 种癌症有没有完美混合在一起！
p2 <- DimPlot(cancer, reduction = "wnn.harmony.umap", group.by = "pancancer_type", label = FALSE) + 
      ggtitle("Pan-Cancer Integration Check")

# 3. 检验sample有没有混合
p3 <- DimPlot(cancer, reduction = "wnn.harmony.umap", group.by = "sample", label = FALSE) + 
      ggtitle("Sample Integration Check")

# 4. 验证细胞类型：看你原来的 celltype 标签是不是聚成了一团
p4 <- DimPlot(cancer, reduction = "wnn.harmony.umap", group.by = "celltype", label = TRUE, repel = TRUE) + 
      ggtitle("Original Celltypes Verification")

# 将图保存到当前目录
ggsave(filename = "1_WNN_Clusters.pdf", plot = p1, width = 12, height = 8)
ggsave(filename = "2_WNN_CancerType.pdf", plot = p2, width = 10, height = 8)
ggsave(filename = "3_WNN_Samples.pdf", plot = p3, width = 15, height = 8)
ggsave(filename = "4_WNN_CellType.pdf", plot = p4, width = 12, height = 8)

In [ ]:
cancer<- FindClusters(
  cancer, 
  graph.name = "wsnn", 
  algorithm = 3,
  resolution = 0.1,
  verbose = FALSE
)

In [ ]:
> table(cancer$seurat_clusters)

    0     1     2     3     4     5     6     7     8     9    10    11    12
61150 55862 28220 14546 14074 12691  7940  7861  4217  3771  1692   828   813
   13    14    15    16    17    18    19    20    21    22    23    24    25
  450     5     4     4     3     3     3     3     3     3     3     3     2
   26    27    28    29    30    31    32    33    34    35    36    37    38
    2     2     2     2     2     2     2     2     2     2     2     2     2
   39    40    41    42    43    44    45    46    47    48    49    50    51
    2     2     2     2     2     2     2     2     2     2     2     2     2
   52    53    54    55    56    57    58    59    60    61    62    63    64
    2     2     2     2     2     2     2     2     2     2     2     2     2
   65    66    67    68    69    70    71    72    73    74    75    76    77
    2     2     2     2     2     2     2     2     2     2     2     2     2
   78    79    80    81    82    83    84    85    86    87    88    89    90
    2     2     2     2     2     2     2     2     2     2     2     2     2
   91    92    93    94    95    96    97    98    99   100   101   102   103
    2     2     2     2     2     2     2     2     2     2     2     2     2
  104   105   106   107   108   109   110   111   112   113   114   115   116
    2     2     2     2     2     2  2825     2     2     2     2     2     2
  117   118   119   120   121   122   123
    2     2     2     2     2     2     2
